# Phase 3 — NYC Property Sales: Research Questions (2003–2023)

This notebook answers the questions in **Initial-Questions.pdf** using the NYC DOF rolling sales dataset.

**Main question:**  
> Which NYC neighborhoods show the most consistent and sustainable property value strength from 2003 to 2023, and how do property classification and location attributes explain that performance?

**Five sub-questions:**
1. Which property *classifications* (residential, mixed-use, commercial) show the strongest long-term price stability?
2. How do *tax classes* (1–4) correlate with pricing stability and growth? Which to prioritize vs avoid?
3. How does physical *property scale* (gross sqft) influence SALE PRICE? What size thresholds separate high-value from low-value tiers?
4. How does *property age* affect market value across boroughs?
5. What physical and structural *characteristics* most strongly explain long-term value?

## How to use this notebook

### Setup (one-time)

Open a PowerShell terminal in this folder (`NYCPS`) and run:

```powershell
python -m venv .venv                       # make an isolated Python environment
.\.venv\Scripts\Activate.ps1               # turn it on (prompt now shows (.venv))
pip install -r requirements.txt            # install pandas, numpy, matplotlib, seaborn, scipy, scikit-learn, jupyter
```

If PowerShell blocks the activation script, run this once:
```powershell
Set-ExecutionPolicy -Scope CurrentUser -ExecutionPolicy RemoteSigned
```

### Running the notebook

* **VSCode** — open this `.ipynb`, click the kernel picker (top-right) and choose `.venv\Scripts\python.exe`.
* **Browser Jupyter** — with the venv active, run `jupyter notebook` and click the file.

Inside the notebook, click a cell and press **Shift + Enter** to run it. Always run cells top-to-bottom the first time — later cells depend on variables defined earlier.

### What to expect when it runs

| Section | What it does | Roughly how long |
|---|---|---|
| 0. Setup & Data Load | Reads the 382 MB CSV into a pandas DataFrame called `df` | 20–40 sec |
| 1. Clean & Derive | Drops bad rows, makes `price_per_sqft`, `sale_year`, `property_age` | 5–10 sec |
| 2. Helper | Defines the `stability_metrics()` function — fast | < 1 sec |
| Q1–Q5 | One section per sub-question; each prints a table and shows a chart | 2–10 sec each |
| ⭐ Main | Ranks every neighborhood and shows the top 20 | 5–10 sec |

### Deliverables for the assignment

1. **This notebook**, fully executed (every cell has output).
2. **A short slide deck** mirroring the style of `Initial-Questions.pdf`: one slide per sub-question showing its chart, one slide with the top-5 neighborhoods, one slide on *why* those neighborhoods won.
3. **Filled-in Conclusions cell** at the bottom — replace the placeholders with the real numbers from your output.

## Methodology — the two numbers we lean on

Almost every question reduces to: *for some grouping (classification, tax class, neighborhood), is the price going up steadily over time?* Two metrics capture that:

**1. CAGR — Compound Annual Growth Rate of median price-per-sqft**  
$$\text{CAGR} = \Big(\frac{P_{\text{last year}}}{P_{\text{first year}}}\Big)^{1/n} - 1$$

Interprets as *"the steady annual % gain that would have produced the same end value."* 5% CAGR over 20 years roughly doubles the price (1.05²⁰ ≈ 2.65×).

**2. CV — Coefficient of Variation across yearly medians**  
$$\text{CV} = \frac{\sigma}{\mu}$$

Standard deviation divided by the mean, computed across the yearly median prices. *Low CV = stable, predictable, boring (good). High CV = the price wobbles year-to-year (risky).*

**Why both?** A neighborhood that went 100 → 1000 → 100 → 1000 has fantastic average CAGR if you pick the right endpoints, but it's awful to invest in. CV catches that. An investment-grade neighborhood has *high CAGR AND low CV* — it grew, and it grew smoothly.

**Why price-per-sqft and not raw price?** A 5,000 sqft townhouse selling for $5M and a 500 sqft studio selling for $500k both equal **$1,000/sqft**. Per-sqft normalizes away the size effect so neighborhoods are comparable.

**Sample-size guardrail.** We only score a group if it has enough sales — 500 for classifications, 1,000 for neighborhoods. A neighborhood with only 12 sales over 20 years gives noisy medians; the threshold filters those out.

## 0. Setup & Data Load

Imports the libraries, then reads the CSV. The path resolves to the local `nyc-property-sales.csv` first; if it's not there it falls back to the old Downloads location.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (11, 5)

CANDIDATE_PATHS = [
    'nyc-property-sales.csv',
    r'C:\Users\Administrator\Downloads\nyc-property-sales.csv',
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), CANDIDATE_PATHS[0])
print('Loading from:', DATA_PATH)

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()
print('Raw shape:', df.shape)
df.head(3)

## 1. Clean & Derive Features

Three things happen here:

* **Type coercion** — SALE PRICE, LAND SQUARE FEET, and GROSS SQUARE FEET arrive as strings with `' -  '` placeholders. `to_numeric(..., errors='coerce')` turns those into `NaN`. SALE DATE becomes a real datetime.
* **Zeros → NaN** — a $0 sale price or 0 sqft is a placeholder, not a real value. Convert them to NaN so they don't poison medians.
* **Derived columns** — the ones every later question needs:
  * `sale_year` (extracted from SALE DATE)
  * `property_age` (sale year − YEAR BUILT)
  * `price_per_sqft` (SALE PRICE ÷ GROSS SQUARE FEET) — the main outcome variable
  * `BOROUGH_NAME` (1→Manhattan, 2→Bronx, …) — friendlier than the integer code
  * `class_group` — collapses the 40+ NYC building-class categories into 5 broad investor-friendly groups

In [ ]:
# Type coercion
for col in ['SALE PRICE', 'LAND SQUARE FEET', 'GROSS SQUARE FEET']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['SALE DATE'] = pd.to_datetime(df['SALE DATE'], errors='coerce')

# Zeros are placeholders
for col in ['SALE PRICE', 'LAND SQUARE FEET', 'GROSS SQUARE FEET', 'YEAR BUILT']:
    df.loc[df[col] == 0, col] = np.nan

BORO = {1: 'Manhattan', 2: 'Bronx', 3: 'Brooklyn', 4: 'Queens', 5: 'Staten Island'}
df['BOROUGH_NAME'] = df['BOROUGH'].map(BORO)

df['sale_year']      = df['SALE DATE'].dt.year
df['property_age']   = df['sale_year'] - df['YEAR BUILT']
df['price_per_sqft'] = df['SALE PRICE'] / df['GROSS SQUARE FEET']

def classify(cat):
    if not isinstance(cat, str):
        return 'Other'
    c = cat.strip().upper()
    if c.startswith(('01', '02', '03', '07', '09', '10', '12', '13', '14', '15', '17')):
        return 'Residential'
    if c.startswith(('04', '08', '11')):
        return 'Mixed-Use / Multifamily'
    if c.startswith(('21', '22', '25', '26', '27', '29', '30')):
        return 'Commercial / Office / Retail'
    if c.startswith(('05', '06', '18', '23', '28')):
        return 'Industrial / Hotel / Specialty'
    if c.startswith(('16', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44')):
        return 'Other / Vacant / Public'
    return 'Other'
df['class_group'] = df['BUILDING CLASS CATEGORY'].apply(classify)

print('Date range:', int(df['sale_year'].min()), '→', int(df['sale_year'].max()))
print('\nRecords by borough:')
print(df['BOROUGH_NAME'].value_counts())

### Valid-sales filter for price analysis

We need a subset that's safe for price/sqft analysis. The filter drops:
* sales under $10,000 (family transfers, deed corrections — not real market trades)
* sales over $200,000,000 (single trophy deals that would dominate medians)
* buildings under 200 sqft or over 1,000,000 sqft (data errors or one-of-a-kind megastructures)
* price/sqft below $20 or above $10,000 (definitely data noise)
* anything outside the 2003–2023 study window

What survives is the working `sales` DataFrame that every Q-section uses.

In [ ]:
valid = (
    df['SALE PRICE'].between(10_000, 200_000_000) &
    df['GROSS SQUARE FEET'].between(200, 1_000_000) &
    df['price_per_sqft'].between(20, 10_000) &
    df['sale_year'].between(2003, 2023)
)
sales = df.loc[valid].copy()
print(f'Valid sales for price analysis: {len(sales):,} of {len(df):,}  ({len(sales)/len(df):.1%})')

## 2. Shared helper — `stability_metrics()`

Every question wants the same four numbers per group:

| Column | Meaning |
|---|---|
| `n_sales` | total transactions (reliability / volume) |
| `median_pps` | overall median price per sqft (price *level*) |
| `cagr_pct` | compound annual growth rate of yearly medians (price *growth*) |
| `cv` | coefficient of variation of yearly medians (price *stability* — lower is better) |

Define it once, reuse it for classification, tax class, and neighborhood.

In [ ]:
def stability_metrics(frame, group_col, min_sales=200):
    yearly = (frame
              .groupby([group_col, 'sale_year'])['price_per_sqft']
              .median()
              .unstack('sale_year'))
    out = pd.DataFrame(index=yearly.index)
    out['n_sales']    = frame.groupby(group_col).size()
    out['median_pps'] = frame.groupby(group_col)['price_per_sqft'].median()
    first = yearly.bfill(axis=1).iloc[:, 0]
    last  = yearly.ffill(axis=1).iloc[:, -1]
    span  = yearly.notna().sum(axis=1).clip(lower=1) - 1
    out['cagr_pct']   = ((last / first) ** (1 / span.replace(0, np.nan)) - 1) * 100
    out['cv']         = (yearly.std(axis=1) / yearly.mean(axis=1)).round(3)
    out = out[out['n_sales'] >= min_sales].sort_values('cagr_pct', ascending=False)
    return out, yearly

## Q1 — Classification × long-term price stability

**Question:** Which broad property type (residential, mixed-use, commercial, industrial) was the most stable from 2003–2023?

**How we answer it:** group the cleaned `sales` by the derived `class_group`, then compute the four metrics. The table is sorted by CAGR; read the `cv` column to see which type was *steady*. Investment-grade = high CAGR with low CV.

In [ ]:
q1_summary, q1_yearly = stability_metrics(sales, 'class_group', min_sales=500)
q1_summary

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for grp, row in q1_yearly.iterrows():
    if grp in q1_summary.index:
        ax.plot(row.index, row.values, marker='o', label=grp)
ax.set_title('Median price-per-sqft by classification, 2003–2023')
ax.set_ylabel('Median $ / sqft'); ax.set_xlabel('Sale year')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

**How to read it:** the line chart shows each classification's median $/sqft year by year. Flat-then-rising lines (smooth) = stable + growing. Sawtooth lines = volatile.  
**Finding (fill after running):** the most stable + growing classification is **\<group with lowest CV and positive CAGR\>**.

## Q2 — Tax classes 1–4: which to prioritize, which to avoid?

NYC DOF tax classes:
* **1** — small residential (1–3 family homes, small condos)
* **2** — larger residential (4+ unit rentals, co-ops, condos)
* **3** — utility property
* **4** — all other commercial / industrial

Same methodology as Q1 — group by `TAX CLASS AT TIME OF SALE`, compute CAGR + CV. The class with the best CAGR-to-CV ratio is the safest investment tier.

In [ ]:
q2_summary, q2_yearly = stability_metrics(sales, 'TAX CLASS AT TIME OF SALE', min_sales=500)
q2_summary

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for tc, row in q2_yearly.iterrows():
    if tc in q2_summary.index:
        ax.plot(row.index, row.values, marker='o', label=f'Tax class {tc}')
ax.set_title('Median $/sqft by tax class')
ax.set_ylabel('Median $ / sqft'); ax.set_xlabel('Sale year')
ax.legend(); plt.tight_layout(); plt.show()

**Rule of thumb:** prioritize classes with `cv < 0.20` *and* `cagr_pct > 3%`. Avoid classes with high CV even if the growth looks attractive — volatility means timing risk.  
**Finding (fill after running):** prioritize tax class **\<x\>**, avoid tax class **\<y\>**.

## Q3 — Size thresholds for high-value vs low-value tiers

**Question:** at what `GROSS SQUARE FEET` does a property cross from "low-value tier" into "high-value tier"?

**Approach:** split all valid sales into five equal-size buckets (quintiles) by gross sqft. For each bucket, compute the sqft range and the median $/sqft. The boundary where median $/sqft *changes regime* (jumps or drops sharply) is the practical threshold.

In [ ]:
sales['size_quintile'] = pd.qcut(sales['GROSS SQUARE FEET'],
                                  q=5,
                                  labels=['Q1 smallest', 'Q2', 'Q3', 'Q4', 'Q5 largest'])
size_summary = (sales.groupby('size_quintile', observed=True)
                .agg(n=('SALE PRICE', 'size'),
                     min_sqft=('GROSS SQUARE FEET', 'min'),
                     max_sqft=('GROSS SQUARE FEET', 'max'),
                     median_price=('SALE PRICE', 'median'),
                     median_pps=('price_per_sqft', 'median')))
size_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
size_summary['median_price'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Median SALE PRICE by size quintile'); axes[0].set_ylabel('$')
axes[0].tick_params(axis='x', rotation=20)
size_summary['median_pps'].plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Median $/sqft by size quintile'); axes[1].set_ylabel('$/sqft')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

**How to read it:** the **left** chart (raw price) always rises monotonically — bigger means pricier. The **right** chart ($/sqft) is the interesting one. A U-shape or step pattern means small *or* huge properties carry a premium per square foot; flat means scale doesn't matter much.  
**Finding (fill after running):** the high-value threshold lands at roughly **\<sqft\>** — properties above this command **\<x\>%** more per sqft than the quintile below.

## Q4 — Property age × borough

**Question:** does newer construction beat older construction, and does the answer depend on which borough?

**Approach:** bin `property_age` into five eras, pivot $/sqft by era × borough, then heatmap. Hot cells = age × borough combinations with the highest $/sqft.

In [ ]:
era_bins   = [-1, 25, 50, 75, 100, 200]
era_labels = ['0–25 yrs', '26–50', '51–75', '76–100', '100+']
sales['age_band'] = pd.cut(sales['property_age'], bins=era_bins, labels=era_labels)

age_pivot = (sales.dropna(subset=['age_band', 'BOROUGH_NAME'])
                  .groupby(['age_band', 'BOROUGH_NAME'], observed=True)['price_per_sqft']
                  .median()
                  .unstack('BOROUGH_NAME'))
age_pivot.round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(age_pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': '$ / sqft'})
ax.set_title('Median $/sqft by property age × borough')
plt.tight_layout(); plt.show()

**Expected pattern:** Manhattan's *oldest* properties (pre-war, 100+ yrs) are often the most expensive per sqft — prestige neighborhoods + architecture premium. Outer boroughs usually reward *newer* construction. The heatmap makes the divergence visible at a glance.

## Q5 — Which structural features explain $/sqft?

**Question:** of all the structural attributes (sqft, units, age, tax class, borough), which ones move price the most?

**Approach:** fit a linear regression on `log(price_per_sqft)`. Logging the target tames the heavy right tail of real-estate prices so coefficients reflect % change instead of $ change. After standardizing features, each coefficient is directly comparable as importance — bigger absolute value = bigger effect.

**Note:** R² will be modest. Real-estate price has a huge unmeasured component (specific block, view, recent renovation) that this dataset doesn't capture. We're not trying to *predict* prices — we're ranking which measurable features explain the most variance.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

features = ['GROSS SQUARE FEET', 'LAND SQUARE FEET',
            'RESIDENTIAL UNITS', 'COMMERCIAL UNITS', 'TOTAL UNITS',
            'property_age']
model_df = sales[features + ['price_per_sqft', 'BOROUGH_NAME', 'TAX CLASS AT TIME OF SALE']].dropna()
model_df = model_df[model_df['TOTAL UNITS'].between(0, 500)]  # cap absurd unit values

X = pd.get_dummies(model_df[features + ['BOROUGH_NAME', 'TAX CLASS AT TIME OF SALE']],
                   columns=['BOROUGH_NAME', 'TAX CLASS AT TIME OF SALE'],
                   drop_first=True)
y = np.log(model_df['price_per_sqft'])

scaler = StandardScaler(with_mean=False)
X_s = scaler.fit_transform(X)
reg = LinearRegression().fit(X_s, y)

coef = pd.Series(reg.coef_, index=X.columns).sort_values(key=abs, ascending=False)
print(f'R² (variance explained): {reg.score(X_s, y):.3f}')
coef.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top = coef.head(12).iloc[::-1]
top.plot(kind='barh', ax=ax,
    color=['steelblue' if v > 0 else 'firebrick' for v in top])
ax.set_title('Standardized feature importance for log($/sqft)')
ax.set_xlabel('Standardized coefficient (positive = pushes price up)')
plt.tight_layout(); plt.show()

**Reading the bar chart:** blue bars push $/sqft up; red bars push it down. The longest bars are the most influential features. Almost always: `BOROUGH_NAME_Manhattan` is the biggest blue bar, confirming location dominates.  
**Finding (fill after running):** the three strongest drivers of $/sqft are **\<feature 1\>**, **\<feature 2\>**, **\<feature 3\>**.

## ⭐ Main question — ranking neighborhoods

**Question:** which NYC neighborhoods showed the most consistent and sustainable value strength from 2003 to 2023?

**Approach:** compute the same `stability_metrics` per neighborhood, then build a composite score that rewards all three things we care about:

$$\text{score} = z(\text{cagr}) - z(\text{cv}) + \tfrac{1}{2}\, z(\log n_\text{sales})$$

* `z(cagr)` rewards growth
* `−z(cv)` rewards stability (subtract volatility)
* `½ z(log n_sales)` gently rewards transaction volume so a noisy 30-sale neighborhood doesn't win

The `z(·)` is z-scoring: subtract the mean, divide by std, so the three components are on the same scale before we add them.

Only neighborhoods with **≥ 1,000 sales** are scored — anything thinner can't be trusted.

In [ ]:
hood, _ = stability_metrics(sales, 'NEIGHBORHOOD', min_sales=1_000)

def z(s): return (s - s.mean()) / s.std()

hood = hood.assign(
    score = z(hood['cagr_pct']) - z(hood['cv']) + z(np.log(hood['n_sales'])) / 2
).sort_values('score', ascending=False)

top20 = hood.head(20)
top20[['n_sales', 'median_pps', 'cagr_pct', 'cv', 'score']].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
top20[['cagr_pct', 'cv', 'score']].plot(kind='barh', ax=ax)
ax.set_title('Top 20 neighborhoods by composite value-strength score (2003–2023)')
ax.invert_yaxis(); plt.tight_layout(); plt.show()

### Why did those neighborhoods win? — cross-tab

Show the *classification* mix inside each of the top-20 neighborhoods and what borough they sit in. The pattern reveals which combination (e.g., "Manhattan + Residential" or "Brooklyn + Mixed-Use") dominates the winners — this is the connective tissue that answers the *and how do property classification and location attributes explain that performance?* part of the main question.

In [ ]:
top_names = top20.index.tolist()
cross = (sales[sales['NEIGHBORHOOD'].isin(top_names)]
         .groupby(['NEIGHBORHOOD', 'BOROUGH_NAME', 'class_group'])
         .size()
         .unstack(fill_value=0))
cross

## 7. Conclusions

Replace each `<…>` placeholder with the real number from your outputs above.

1. **Most stable classification (Q1):** `<class_group>` — CAGR `<x>%`, CV `<y>`.
2. **Tax-class recommendation (Q2):** prioritize tax class `<n>`; avoid tax class `<m>`. Reason: `<one sentence>`.
3. **Size threshold (Q3):** the high-value tier kicks in above `<sqft>` gross sqft, where $/sqft jumps by ~`<x>%` relative to the quintile below.
4. **Age × borough (Q4):** in `<borough>`, the `<age band>` band wins at `<$/sqft>`; in `<other borough>` it's `<age band>` that wins. Pattern: `<one sentence>`.
5. **Top structural drivers (Q5):** the three most impactful standardized coefficients are `<feature 1>`, `<feature 2>`, `<feature 3>`. Combined R² = `<x>`.
6. **Top 5 neighborhoods (main question):** `<name 1>`, `<name 2>`, `<name 3>`, `<name 4>`, `<name 5>`. The common thread: `<borough(s)>` + `<class_group(s)>` + `<tax class(es)>`.

### Deliverables checklist
- [ ] Notebook runs top-to-bottom without errors
- [ ] Each Q-section's table and chart are visible
- [ ] Placeholders above are replaced with real numbers
- [ ] Slide deck built — one slide per sub-question, one for the top-5 neighborhoods, one explaining *why* they won